# Delaware — Title 18 (Insurance Code) → `data/delaware/ins_codes/*.md`

Delaware’s **Insurance Code** is **Title 18** of the **Delaware Code**, published at **[delcode.delaware.gov/title18/](https://delcode.delaware.gov/title18/)**.

Each **chapter** (and some **subchapters**) is an **`index.html`** page. Statutes are **`div.Section`** blocks: a **`div.SectionHead`** with **`id`** equal to the section number (e.g. `701`, `3550`) supplies the heading; the rest of the **`div.Section`** is the body.

This notebook:

1. Loads the Title 18 landing page and **BFS**-collects every **`…/title18/cNNN/index.html`** and **`…/title18/cNNN/scMM/index.html`** index URL.
2. Fetches each index page and writes one **Markdown** file per section: **`DE_sec_<path>_<id>.md`** (e.g. `DE_sec_c007_701.md`, `DE_sec_c035_sc03_3550.md`).

**Official source:** each file cites **`chapter…/index.html#<id>`**.

**Volume:** on the order of **1,600** sections across **~130** index pages — a full run takes a few minutes. Use **`MAX_SECTIONS`** to cap exports while testing.

**Politeness:** **`REQUEST_DELAY_SEC`** between HTTP requests.

**SSL:** If you hit certificate verification errors, set **`VERIFY_SSL = False`** in the config cell (less secure) or keep **`certifi`** up to date.

Then run **`python -m app.ingest`** from the project root.


In [1]:
%pip install -q httpx beautifulsoup4 certifi


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from pathlib import Path
from urllib.parse import urljoin

import certifi
import httpx
from bs4 import BeautifulSoup

TITLE_ROOT = "https://delcode.delaware.gov/title18/"
OUT_DIR = Path("data") / "delaware" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

USER_AGENT = "RAG-DE-Title18/1.0 (public Delaware Code; educational indexing)"
REQUEST_DELAY_SEC = 0.2
TIMEOUT = 90.0

MAX_SECTIONS = 0
SKIP_EXISTING = True

VERIFY_SSL = True
_verify = certifi.where() if VERIFY_SSL else False

INDEX_RE = re.compile(r"/title18/(c\d+(?:/sc\d+)?)/index\.html$", re.I)


In [3]:
def fetch_html(client: httpx.Client, url: str) -> str:
    r = client.get(url, follow_redirects=True)
    r.raise_for_status()
    time.sleep(REQUEST_DELAY_SEC)
    return r.text


def extract_index_links(html: str, base_url: str) -> list[str]:
    soup = BeautifulSoup(html, "html.parser")
    out: list[str] = []
    for a in soup.find_all("a", href=True):
        absu = urljoin(base_url, a["href"]).split("#", 1)[0]
        if INDEX_RE.search(absu.replace("https://delcode.delaware.gov", "")):
            out.append(absu)
    return list(dict.fromkeys(out))


def page_slug_from_url(index_url: str) -> str:
    m = INDEX_RE.search(index_url.replace("https://delcode.delaware.gov", ""))
    return m.group(1).replace("/", "_") if m else "title18"


def slug_sort_key(slug: str) -> tuple:
    m = re.match(r"c(\d+)(?:_sc(\d+))?", slug, re.I)
    if not m:
        return (0, 0)
    return (int(m.group(1)), int(m.group(2) or 0))


def section_id_sort_key(sec_id: str) -> tuple:
    if sec_id.isdigit():
        return (0, int(sec_id))
    m = re.match(r"(\d+)", sec_id)
    return (1, int(m.group(1))) if m else (2, sec_id)


def parse_sections_from_index(html: str, index_url: str) -> list[tuple[str, str, str, str]]:
    """Return [(page_slug, section_id, body_text, official_url), ...]."""
    slug = page_slug_from_url(index_url)
    soup = BeautifulSoup(html, "html.parser")
    rows: list[tuple[str, str, str, str]] = []
    base = index_url.split("#", 1)[0]
    for sec in soup.find_all("div", class_="Section"):
        head = sec.find("div", class_="SectionHead")
        if not head or not head.get("id"):
            continue
        sid = head["id"].strip()
        body = sec.get_text("\n", strip=True)
        official = f"{base}#{sid}"
        rows.append((slug, sid, body, official))
    return rows


def section_to_filename(page_slug: str, sec_id: str) -> str:
    safe_id = re.sub(r"[^0-9a-zA-Z]+", "_", sec_id).strip("_")
    return f"DE_sec_{page_slug}_{safe_id}.md"


def crawl_title18() -> list[tuple[str, str, str, str, tuple]]:
    """Return rows with sort key tuple for ordering."""
    out: list[tuple[str, str, str, str, tuple]] = []
    with httpx.Client(
        headers={"User-Agent": USER_AGENT, "Accept": "text/html,*/*;q=0.8"},
        timeout=TIMEOUT,
        verify=_verify,
        http2=False,
    ) as client:
        r0 = client.get(TITLE_ROOT, follow_redirects=True)
        r0.raise_for_status()
        time.sleep(REQUEST_DELAY_SEC)
        root_final = str(r0.url)
        seed = extract_index_links(r0.text, root_final)
        seen: set[str] = set()
        queue = list(seed)
        discovered: list[str] = []
        while queue:
            url = queue.pop(0)
            if url in seen:
                continue
            seen.add(url)
            discovered.append(url)
            html = fetch_html(client, url)
            for row in parse_sections_from_index(html, url):
                slug, sid, body, official = row
                sk = (slug_sort_key(slug), section_id_sort_key(sid))
                out.append((slug, sid, body, official, sk))
            for nxt in extract_index_links(html, url):
                if nxt not in seen and nxt not in queue:
                    queue.append(nxt)
        (OUT_DIR / "_delaware_title18_index_urls.txt").write_text(
            "\n".join(discovered), encoding="utf-8"
        )
    out.sort(key=lambda r: r[4])
    dedup: dict[tuple[str, str], tuple[str, str, str, tuple]] = {}
    for slug, sid, body, official, sk in out:
        key = (slug, sid)
        if key not in dedup:
            dedup[key] = (body, official, sk)
    merged: list[tuple[str, str, str, str, tuple]] = []
    for slug, sid in sorted(dedup.keys(), key=lambda k: (slug_sort_key(k[0]), section_id_sort_key(k[1]))):
        body, official, sk = dedup[(slug, sid)]
        merged.append((slug, sid, body, official, sk))
    return merged


def download_title18() -> dict[str, int]:
    rows = crawl_title18()
    print(f"Discovered {len(rows)} sections from Title 18 index pages")

    todo = rows if not MAX_SECTIONS else rows[:MAX_SECTIONS]
    if MAX_SECTIONS:
        print(f"Limited export to first {len(todo)} sections (MAX_SECTIONS)")

    wrote, skipped = 0, 0
    for slug, sid, body, official, _sk in todo:
        dest = OUT_DIR / section_to_filename(slug, sid)
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
            skipped += 1
            continue
        title = f"Delaware Code — Title 18 (Insurance) § {sid}"
        md = (
            f"# {title}\n\n"
            f"**Delaware Code — Title 18 (Insurance Code)**\n\n"
            f"**Official source:** {official}\n\n"
            f"**Chapter / subchapter (URL slug):** {slug.replace('_', '/')}\n\n"
            f"**Section id:** {sid}\n\n"
            f"---\n\n"
            f"{body}\n"
        )
        dest.write_text(md, encoding="utf-8")
        wrote += 1

    print(f"Done. wrote={wrote} skipped={skipped} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped}


download_title18()


Discovered 1586 sections from Title 18 index pages
Done. wrote=1586 skipped=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/delaware/ins_codes


{'wrote': 1586, 'skipped': 0}

## Next step

`python -m app.ingest` from the project root.
